# BigAlpha 2026 端到端模型推理（双频 concat）

本 notebook 是唯一推理 notebook，平台调用 `main()`。
模型训练由 `transformer_train.py` 完成，训练产物为 `transformer_model.json`。

In [ ]:
"""BigAlpha 端到端模型全局配置。提交要求的超参配置 + 随机种子集中于此。"""
import random

import numpy as np

# 字段清单依据平台实际 schema (bar15m / bar30m, 盘口 5 档,
# 无 num_trades/avg_price/total_volume 系字段; 成交笔数为 deal_number)。
PRICE_COLS = ["open", "high", "low", "close", "pre_close",
              "bid_price1", "bid_price2", "bid_price3", "bid_price4", "bid_price5",
              "ask_price1", "ask_price2", "ask_price3", "ask_price4", "ask_price5"]
VOL_COLS = ["volume", "amount", "deal_number",
            "bid_volume1", "bid_volume2", "bid_volume3", "bid_volume4", "bid_volume5",
            "ask_volume1", "ask_volume2", "ask_volume3", "ask_volume4", "ask_volume5"]

CONFIG = {
    "table": "bigalpha_2026_stock_bar15m",          # 高频内生表 (16 bar/天)
    "lowfreq_table": "bigalpha_2026_stock_bar30m",  # 低频外生表 (8 bar/天)
    "lowfreq_mode": "concat",  # "concat"—双频 bar 拼接走单一 IntradayEncoder
    "instruments_table": "bigalpha_2026_instruments",
    "exposure_table": "bigalpha_2026_exposure",
    "bars_per_day": 16,          # 高频 (15m) 日内 bar 数
    "lowfreq_bars_per_day": 8,   # 低频 (30m) 日内 bar 数
    "price_cols": PRICE_COLS,
    "vol_cols": VOL_COLS,
    "feature_cols": PRICE_COLS + VOL_COLS,
    # 复权因子仅用于标签收益计算 (close*adjust_factor), 不作为模型输入字段
    "adjust_col": "adjust_factor",
    "lookback_days": 60,
    "infer_buffer_natural_days": 130,
    "chunk_size": 250,  # 每块股票数; 内存紧张(<32G)时降回 100
    "train_frac": 0.8,
    "seed": 456,
    "tag": "submission",
    "cache_path": "train_cache.npz",
    "cache_path_low": "train_cache_low.npz",
    "checkpoint_path": "checkpoint.pt",
    "resume_checkpoint_path": "last_checkpoint.pt",
    # 官方提交的训练产物必须为 JSON；checkpoint.pt 仅用于本地断点/最佳权重。
    "artifact_path": "transformer_model.json",
    "label": {"mode": "residual", "horizon": 1, "winsor_pct": 1.0},
    "model": {"d_intra": 128, "heads_intra": 8, "layers_intra": 4, "ffn_intra": 512,
              "intra_chunk_size": 8192,
              "d_day": 256, "tau_cross": 8, "heads_cross": 4, "ffn_cross": 1024,
              "use_cross_attn": True, "gate_init": -5.0, "glu_bottleneck": 128,
              "dropout": 0.1,
              "n_global_tokens": 4},
    "loss": {"mse_w": 0.01, "listnet_w": 0.1, "listnet_temp": 1.0,
             "listmle_w": 0.1, "listmle_temp": 1.0,
             "min_pred_std": 0.05, "std_w": 1.0},
    "train": {"epochs": 40, "lr": 5e-4, "weight_decay_head": 0.01, "clip": 1.0,
              "patience": 10, "swa_frac": 0.9, "stock_sample_ratio": 1.0,
              "log_every": 5, "checkpoint_every": 100,
              "time_budget_min": 160,  # 2h40m, 留 20min 余量应对平台 3h 硬限制
              "metric_w": {"ic": 1.0, "icir": 0.02, "sharpe": 0.02}},
}

PARAM_MIN, PARAM_MAX = 100_000, 100_000_000


def validate_config(cfg):
    assert len(cfg["feature_cols"]) <= 100, "字段数超过赛规上限 100"
    assert len(cfg["feature_cols"]) == len(set(cfg["feature_cols"])), "字段重复"
    assert cfg["lookback_days"] <= 240, "回看窗口超过赛规上限 240 交易日"
    assert set(cfg["price_cols"]).isdisjoint(cfg["vol_cols"])
    assert cfg.get("resume_checkpoint_path", "last_checkpoint.pt") != \
        cfg["checkpoint_path"], "断点文件与 best model 必须使用不同路径"


def assert_param_count(n):
    assert PARAM_MIN <= n <= PARAM_MAX, \
        f"参数量 {n} 超出赛规区间 [{PARAM_MIN}, {PARAM_MAX}]"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

In [ ]:
"""数据层：合规预处理、流式统计、按日透视、分块取数与缓存。

平台查询通过可注入的 query_fn(sql, filters) -> DataFrame 隔离,
平台侧实现为 lambda sql, filters: dai.query(sql, filters=filters, compression=True).df()。
读取方式遵循官方 OOM 指引: 按股票分块、只 SELECT 所需列、逐块释放。
"""
import numpy as np
import pandas as pd


class RunningStats:
    """流式 mean/std（官方 OOM 指引方案），与全量一次计算数值等价。"""

    def __init__(self, n_feat):
        self.n = 0
        self.s = np.zeros(n_feat, np.float64)
        self.ss = np.zeros(n_feat, np.float64)

    def update(self, x):
        x = x.reshape(-1, x.shape[-1]).astype(np.float64)
        self.n += x.shape[0]
        self.s += x.sum(0)
        self.ss += (x ** 2).sum(0)

    def finalize(self):
        mean = self.s / self.n
        var = self.ss / self.n - mean ** 2
        std = np.sqrt(np.clip(var, 0, None)) + 1e-6
        return mean.astype(np.float32), std.astype(np.float32)


def apply_field_transforms(df, cfg):
    """按字段统一变换（赛规允许类）：价格 log、量 log1p。"""
    out = df.copy()
    for c in cfg["price_cols"]:
        if c in out:
            out[c] = np.log(out[c].clip(lower=1e-6))
    for c in cfg["vol_cols"]:
        if c in out:
            out[c] = np.log1p(out[c].clip(lower=0))
    return out


class Normalizer:
    def __init__(self, mean, std):
        self.mean = np.asarray(mean, np.float32)
        self.std = np.asarray(std, np.float32)

    def apply(self, x):
        return ((x - self.mean) / self.std).astype(np.float32)


def pivot_to_days(df, feature_cols, bars_per_day, close_col="close"):
    """单只股票的 bar 级 df (按 date 升序) -> (dates (D,), X (D,B,F), day_close (D,))。
    向量化实现: 整只股票一次 ffill/bfill (缺失值填充), 按日期边界 numpy 切片;
    缺 bar 的天以当日首 bar 左侧补齐, 多余 bar 取最后 bars_per_day 根。"""
    n_feat = len(feature_cols)
    empty = (np.array([], "datetime64[D]"),
             np.zeros((0, bars_per_day, n_feat), np.float32),
             np.array([], np.float64))
    if len(df) == 0:
        return empty
    feats = df[feature_cols].to_numpy(np.float32)
    feats = pd.DataFrame(feats).ffill().bfill().to_numpy(np.float32)
    closes_all = df[close_col].to_numpy(np.float64)
    day = df["date"].dt.normalize().to_numpy().astype("datetime64[D]")
    starts = np.flatnonzero(np.concatenate([[True], day[1:] != day[:-1]]))
    ends = np.append(starts[1:], len(day))
    counts = ends - starts

    # 快速路径: 所有天都是标准 bar 数且无缺失
    if np.all(counts == bars_per_day) and np.isfinite(feats).all():
        return (day[starts],
                feats.reshape(-1, bars_per_day, n_feat),
                closes_all[ends - 1])

    dates, arrs, closes = [], [], []
    for s, e in zip(starts, ends):
        a = feats[s:e]
        if not np.isfinite(a).all():
            continue
        if len(a) >= bars_per_day:
            a = a[-bars_per_day:]
        else:
            a = np.concatenate([np.repeat(a[:1], bars_per_day - len(a), axis=0), a])
        dates.append(day[s])
        arrs.append(a)
        closes.append(closes_all[e - 1])
    if not dates:
        return empty
    return np.array(dates), np.stack(arrs), np.array(closes, np.float64)


def iter_chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def list_instruments(query_fn, table, start, end):
    df = query_fn(f"SELECT DISTINCT instrument FROM {table}",
                  {"date": [start, end]})
    return sorted(df["instrument"].tolist())


def load_stock_arrays(query_fn, table, cfg, start, end, instruments,
                     bars_per_day=None):
    """分块查询 -> 合规变换 -> 按日透视。每块用完即释放（官方 OOM 指引）。
    day_close 用于标签: 若表提供复权因子 (adjust_col), 用 close*adjust_factor
    计算复权收益, 避免分红送转污染标签; 复权因子不进入模型输入。"""
    import time as _time
    bpd = bars_per_day if bars_per_day is not None else cfg["bars_per_day"]
    adj = cfg.get("adjust_col")
    extra = [adj] if adj and adj not in cfg["feature_cols"] else []
    cols = ", ".join(["date", "instrument"] + cfg["feature_cols"] + extra)
    sql = f"SELECT {cols} FROM {table} ORDER BY instrument, date"
    out = {}
    chunks = list(iter_chunks(instruments, cfg["chunk_size"]))
    t_start = _time.time()
    for ci, chunk in enumerate(chunks):
        t_c = _time.time()
        df = query_fn(sql, {"date": [start, end], "instrument": list(chunk)})
        n_rows = 0 if df is None else len(df)
        elapsed = _time.time() - t_start
        eta = elapsed / (ci + 1) * (len(chunks) - ci - 1)
        table_tag = table.rsplit("_", 1)[-1] if "_" in table else table
        print(f"[data:{table_tag}] chunk {ci + 1}/{len(chunks)} rows={n_rows} "
              f"query={_time.time() - t_c:.0f}s elapsed={elapsed:.0f}s "
              f"eta={eta:.0f}s", flush=True)
        if df is None or len(df) == 0:
            continue
        t_p = _time.time()
        if adj and adj in df.columns:
            raw_close = df["close"].astype(np.float64) * df[adj].astype(np.float64)
        else:
            raw_close = df["close"].copy()
        df = apply_field_transforms(df, cfg)
        df["_raw_close"] = raw_close
        for ins, sub in df.groupby("instrument", sort=False):
            dates, X, close = pivot_to_days(sub, cfg["feature_cols"],
                                            bpd, close_col="_raw_close")
            if len(dates):
                out[str(ins)] = (dates, X.astype(np.float16), close)
        print(f"[data:{table_tag}] chunk {ci + 1}/{len(chunks)} processed "
              f"proc={_time.time() - t_p:.0f}s stocks_total={len(out)}", flush=True)
        del df
    return out


def save_cache(path, arrays):
    import time as _time
    t0 = _time.time()
    print(f"[cache] saving {len(arrays)} stocks -> {path} ...", flush=True)
    flat = {}
    for ins, (dates, X, close) in arrays.items():
        flat[f"{ins}::dates"] = dates
        flat[f"{ins}::X"] = X
        flat[f"{ins}::close"] = close
    # 不压缩: float16 行情数据压缩比低, savez_compressed 单线程要数分钟。
    # 先写临时文件再原子改名: 中断不会在目标路径留下残缺文件。
    import os as _os
    tmp = path + ".tmp.npz"
    np.savez(tmp, **flat)
    _os.replace(tmp, path)
    print(f"[cache] saved in {_time.time() - t0:.0f}s", flush=True)


def load_cache(path):
    z = np.load(path, allow_pickle=False)
    names = sorted({k.split("::")[0] for k in z.files})
    return {n: (z[f"{n}::dates"], z[f"{n}::X"], z[f"{n}::close"]) for n in names}

In [ ]:
"""模型：双频架构 (15m 内生 + 30m 外生 concat)，约 220 万参数。

字段混合 -> 日内编码(ALiBi+global token) -> 日间GRU -> 逐时刻截面注意力(门控残差)
-> 末日query时序聚合 -> RankGLU打分头。
"""
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint as _ckpt


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def alibi_bias(n_heads, length, device):
    slopes = torch.tensor([2 ** (-8.0 * (i + 1) / n_heads) for i in range(n_heads)],
                          device=device)
    pos = torch.arange(length, device=device)
    dist = (pos[None, :] - pos[:, None]).abs().float()
    return -slopes[:, None, None] * dist[None]


class FieldMix(nn.Module):
    """字段维混合块 (StockMixer): 端到端学跨字段交互, 替代被禁止的人工跨字段算子。"""

    def __init__(self, n_feat):
        super().__init__()
        self.norm = nn.LayerNorm(n_feat)
        self.fc1 = nn.Linear(n_feat, n_feat)
        self.fc2 = nn.Linear(n_feat, n_feat)
        self.act = nn.Hardswish()

    def forward(self, x):
        return x + self.fc2(self.act(self.fc1(self.norm(x))))


class _EncoderLayer(nn.Module):
    def __init__(self, d, heads, ffn, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, ffn), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(ffn, d))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        h = self.n1(x)
        a, _ = self.attn(h, h, h, attn_mask=mask, need_weights=False)
        x = x + self.drop(a)
        return x + self.drop(self.ffn(self.n2(x)))


class IntradayEncoder(nn.Module):
    """日内 bar 序列 -> 当日向量。ALiBi 距离衰减偏置 (TIPS) + global token 聚合 (TimeXer)。"""

    def __init__(self, n_feat, d, heads, layers, ffn, bars, dropout):
        super().__init__()
        self.proj = nn.Linear(n_feat, d)
        self.pos = nn.Parameter(torch.zeros(1, bars, d))
        self.glb = nn.Parameter(torch.zeros(1, 1, d))
        self.layers = nn.ModuleList(_EncoderLayer(d, heads, ffn, dropout)
                                    for _ in range(layers))
        self.heads = heads
        self.out_norm = nn.LayerNorm(d)
        bias = torch.zeros(heads, bars + 1, bars + 1)
        bias[:, 1:, 1:] = alibi_bias(heads, bars, "cpu")
        self.register_buffer("attn_bias", bias, persistent=False)

    def forward(self, x):  # (B*, bars, F) -> (B*, d)
        h = self.proj(x) + self.pos
        h = torch.cat([self.glb.expand(h.shape[0], -1, -1), h], dim=1)
        mask = self.attn_bias.repeat(h.shape[0], 1, 1)
        for lyr in self.layers:
            h = lyr(h, mask)
        return self.out_norm(h[:, 0])


class VariateEncoder(nn.Module):
    """低频外生变量编码器 (TimeXer variate-wise):
    每个特征 → 1 个 token (不依赖时序长度), 特征间自注意力。"""

    def __init__(self, n_feat, bars, d, dropout):
        super().__init__()
        self.n_feat = n_feat
        self.bars = bars
        self.proj = nn.Linear(bars, d)       # 每个特征的整条 bar 序列 → d
        self.feat_embed = nn.Parameter(torch.zeros(1, n_feat, d))
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):  # (B*, bars, F) -> (B*, F, d)
        h = x.transpose(1, 2)                # (B*, F, bars)
        h = self.proj(h) + self.feat_embed   # (B*, F, d)
        return self.drop(self.norm(h))


class CrossFreqBridge(nn.Module):
    """可学习全局 token 通过 cross-attention 查询外生 feature token (TimeXer 桥接)。"""

    def __init__(self, d, heads, n_global=4, dropout=0.1):
        super().__init__()
        self.global_tokens = nn.Parameter(torch.zeros(1, n_global, d))
        self.cross_attn = nn.MultiheadAttention(d, heads, dropout=dropout,
                                                batch_first=True)
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)
        nn.init.trunc_normal_(self.global_tokens, std=0.02)

    def forward(self, feat_tokens):  # (B*, F, d) -> (B*, d)
        B = feat_tokens.shape[0]
        g = self.global_tokens.expand(B, -1, -1)           # (B*, n_global, d)
        enriched, _ = self.cross_attn(query=g, key=feat_tokens, value=feat_tokens)
        enriched = self.norm(g + self.drop(enriched))
        return enriched.mean(1)                            # pool → (B*, d)


class InterDayEncoder(nn.Module):
    def __init__(self, d_in, d):
        super().__init__()
        self.gru = nn.GRU(d_in, d, batch_first=True)

    def forward(self, x):  # (N, L, d_in) -> (N, L, d)
        out, _ = self.gru(x)
        return out


class CrossSectionBlock(nn.Module):
    """截面注意力: batch 维=时间位置, 序列维=股票; 无位置编码 (排列不变, iTransformer);
    门控残差近零初始化 (WaveLSFormer): 先学单股基线, 训练中逐步启用截面信息。"""

    def __init__(self, d, heads, ffn, dropout, gate_init):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.gate_attn = nn.Parameter(torch.tensor(float(gate_init)))
        self.ffn_norm = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, ffn), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(ffn, d))
        self.gate_ffn = nn.Parameter(torch.tensor(float(gate_init)))

    def forward(self, h):  # (tau, N, d) -> (tau, N, d)
        x = self.norm(h)
        a, _ = self.attn(x, x, x, need_weights=False)
        h = h + torch.sigmoid(self.gate_attn) * a
        return h + torch.sigmoid(self.gate_ffn) * self.ffn(self.ffn_norm(h))


class TemporalAggregator(nn.Module):
    """以最新一天为 query 的时序注意力加权 (MASTER/DTML)。
    打分全程 fp32: fp16 下 q·k 内积可溢出为 ±inf, softmax 内 inf-inf → NaN
    (epoch 17 起逐股触发的根因, 由 code/diagnose_nan_layer.py 定位)。"""

    def __init__(self, d):
        super().__init__()
        self.q, self.k = nn.Linear(d, d), nn.Linear(d, d)
        self.scale = d ** -0.5

    def forward(self, h):  # (N, tau, d) -> (N, d)
        with torch.autocast(device_type=h.device.type, enabled=False):
            h = h.float()
            w = torch.softmax((self.q(h[:, -1:]) @ self.k(h).transpose(1, 2))
                              * self.scale, -1)
            return (w @ h).squeeze(1)


class RankGLUHead(nn.Module):
    """线性直通 + γ·瓶颈 GLU 门控残差 (RankGLU): 稳定排序几何 + 有界低秩非线性。"""

    def __init__(self, d, b, gamma=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.lin = nn.Linear(d, 1)
        self.v, self.g = nn.Linear(d, b), nn.Linear(d, b)
        self.out = nn.Linear(b, 1)
        self.gamma = nn.Parameter(torch.tensor(float(gamma)))

    def forward(self, e):
        e = self.norm(e)
        z = self.v(e) * torch.sigmoid(self.g(e))
        return (self.lin(e) + self.gamma * self.out(z)).squeeze(-1)


class StockScorer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        m = cfg["model"]
        n_feat = len(cfg["feature_cols"])
        self.tau = m["tau_cross"]
        self.use_cross = m["use_cross_attn"]
        self.intra_chunk_size = m.get("intra_chunk_size", 4096)
        self.has_lowfreq = cfg.get("lowfreq_table") is not None
        self.lowfreq_mode = cfg.get("lowfreq_mode", "cross_attn") if self.has_lowfreq else None
        d = m["d_intra"]

        if self.has_lowfreq and self.lowfreq_mode == "concat":
            # concat 模式：把两频 bar 拼成一个更宽的日内序列，走单一 IntradayEncoder
            total_bars = cfg["bars_per_day"] + cfg["lowfreq_bars_per_day"]
            self.fieldmix = FieldMix(n_feat)
            self.intra = IntradayEncoder(n_feat, d, m["heads_intra"],
                                         m["layers_intra"], m["ffn_intra"],
                                         total_bars, m["dropout"])
            self.fuse = None  # 不需要融合层
        else:
            self.fieldmix = FieldMix(n_feat)
            self.intra = IntradayEncoder(n_feat, d, m["heads_intra"],
                                         m["layers_intra"], m["ffn_intra"],
                                         cfg["bars_per_day"], m["dropout"])

            if self.has_lowfreq:
                lb = cfg["lowfreq_bars_per_day"]
                self.fieldmix_low = FieldMix(n_feat)
                self.variate = VariateEncoder(n_feat, lb, d, m["dropout"])
                self.bridge = CrossFreqBridge(d, m["heads_intra"],
                                              m.get("n_global_tokens", 4),
                                              m["dropout"])
                self.fuse = nn.Linear(d * 2, d)
            else:
                self.fuse = None

        self.inter = InterDayEncoder(d, m["d_day"])
        self.cross = CrossSectionBlock(m["d_day"], m["heads_cross"], m["ffn_cross"],
                                       m["dropout"], m["gate_init"])
        self.agg = TemporalAggregator(m["d_day"])
        self.head = RankGLUHead(m["d_day"], m["glu_bottleneck"])

    def encode_days(self, x, x_low=None):  # -> (N, L, d_day)
        N, L, B, F = x.shape

        # ── concat 模式：双频 bar 直接拼成宽序列 ──
        if self.has_lowfreq and self.lowfreq_mode == "concat":
            if x_low is not None:
                _, _, BL, _ = x_low.shape
                x = torch.cat([x, x_low], dim=2)   # (N, L, B+BL, F)
                B = B + BL
            else:
                # 低频数据缺失时用零补齐，保持 encoder 期望的 bar 数不变
                extra = self.cfg.get("lowfreq_bars_per_day", 0)
                if extra:
                    pad = torch.zeros(N, L, extra, F, device=x.device, dtype=x.dtype)
                    x = torch.cat([x, pad], dim=2)
                    B = B + extra

        # ── 高频路径（内生）/ concat 后的统一路径 ──
        flat = x.reshape(N * L, B, F)
        chunks = []
        for start in range(0, len(flat), self.intra_chunk_size):
            part = flat[start:start + self.intra_chunk_size]

            def _fn(p):
                return self.intra(self.fieldmix(p))

            chunks.append(_ckpt(_fn, part, use_reentrant=False))
        daily = torch.cat(chunks, dim=0)  # (N*L, d_intra)

        # ── 低频路径（外生, TimeXer variate-wise, 仅 cross_attn 模式）──
        if (self.has_lowfreq and self.lowfreq_mode != "concat"
                and x_low is not None):
            _, _, BL, _ = x_low.shape
            flat_low = x_low.reshape(N * L, BL, F)
            chunks_low = []
            for start in range(0, len(flat_low), self.intra_chunk_size):
                part = flat_low[start:start + self.intra_chunk_size]

                def _fn_low(p):
                    return self.variate(self.fieldmix_low(p))

                chunks_low.append(_ckpt(_fn_low, part, use_reentrant=False))
            feat_tokens = torch.cat(chunks_low, dim=0)  # (N*L, F, d_intra)
            context = self.bridge(feat_tokens)            # (N*L, d_intra)
            daily = self.fuse(torch.cat([daily, context], dim=-1))  # (N*L, d_intra)

        return self.inter(daily.reshape(N, L, -1))

    def score_day(self, H):  # (N, tau, d_day) -> (N,)
        if self.use_cross:
            H = self.cross(H.transpose(0, 1)).transpose(0, 1)
        return self.head(self.agg(H))

    def forward(self, x, x_low=None):
        H = self.encode_days(x, x_low)
        return self.score_day(H[:, -self.tau:])

In [ ]:
"""推理：分块编码 -> 按日截面打分 -> instruments 骨架 left join。

构造上保证平台三项校验必过: 三列名精确匹配、评估区间交易日完整、逐日缺失率为 0
（无法计算的 (date, instrument) 填入当日截面中位数）。
"""
import numpy as np
import pandas as pd
import torch


def run_inference(ckpt, query_fn, infer_table, instruments_query_fn,
                  start_date, end_date, device=None, lowfreq_table=None):
    cfg = ckpt["config"]
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = StockScorer(cfg).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    if not cfg.get("skip_param_check"):
        assert_param_count(count_params(model))
    normalizer = Normalizer(ckpt["mean"], ckpt["std"])

    skeleton = instruments_query_fn(start_date, end_date)[["date", "instrument"]].copy()
    skeleton["date"] = pd.to_datetime(skeleton["date"]).dt.normalize()
    instruments = sorted(skeleton["instrument"].unique().tolist())

    buf = (pd.to_datetime(start_date)
           - pd.Timedelta(days=cfg["infer_buffer_natural_days"])).strftime("%Y-%m-%d 00:00:00")
    sd = pd.to_datetime(start_date).normalize()
    ed = pd.to_datetime(end_date).normalize()

    # 低频表优先用外部传入（datasources 注入），其次用 CONFIG 硬编码名
    lf_table = lowfreq_table or cfg.get("lowfreq_table")
    has_lowfreq = lf_table is not None

    tau = cfg["model"]["tau_cross"]
    per_day = {}  # date -> list[(instrument, H_tau (tau, d_day))]
    with torch.no_grad():
        for chunk in iter_chunks(instruments, cfg["chunk_size"]):
            arrays_high = load_stock_arrays(query_fn, infer_table, cfg,
                                            buf, str(end_date), list(chunk),
                                            bars_per_day=cfg["bars_per_day"])
            arrays_low = None
            if has_lowfreq:
                arrays_low = load_stock_arrays(query_fn, lf_table, cfg,
                                               buf, str(end_date), list(chunk),
                                               bars_per_day=cfg["lowfreq_bars_per_day"])
            # 按 instrument 对齐双频数据
            for ins in arrays_high:
                dates_high, X_high, _ = arrays_high[ins]
                X_low = None
                if arrays_low and ins in arrays_low:
                    dates_low, X_low_raw, _ = arrays_low[ins]
                idx = [i for i, d in enumerate(dates_high)
                       if sd <= pd.Timestamp(d) <= ed]
                if not idx:
                    continue
                Xn = torch.from_numpy(
                    normalizer.apply(X_high.astype(np.float32)))[None].to(device)
                Xn_low = None
                if X_low is not None:
                    # 对齐低频日期：找到每个高频评估日对应的低频日位置
                    low_date_to_i = {pd.Timestamp(d): i for i, d in enumerate(dates_low)}
                    low_indices = []
                    for i in idx:
                        d_high = pd.Timestamp(dates_high[i])
                        li = low_date_to_i.get(d_high)
                        if li is not None:
                            low_indices.append(li)
                    if low_indices:
                        X_low_aligned = X_low_raw[low_indices]
                        Xn_low = torch.from_numpy(
                            normalizer.apply(X_low_aligned.astype(np.float32)))[None].to(device)
                if Xn_low is not None:
                    H = model.encode_days(Xn, Xn_low)[0].cpu()  # (D, d_day)
                else:
                    H = model.encode_days(Xn)[0].cpu()  # (D, d_day)
                for i in idx:
                    h = H[max(0, i + 1 - tau): i + 1]
                    if len(h) < tau:  # 历史不足: 重复首日补齐
                        h = torch.cat([h[:1].expand(tau - len(h), -1), h])
                    d_dt = pd.Timestamp(dates_high[i])
                    per_day.setdefault(d_dt, []).append((ins, h))
            del arrays_high
            if arrays_low:
                del arrays_low

        rows = []
        for d, items in per_day.items():
            Ht = torch.stack([h for _, h in items]).to(device)  # (N, tau, d_day)
            scores = model.score_day(Ht).cpu().numpy()
            rows += [(d, ins, float(s)) for (ins, _), s in zip(items, scores)]

    pred = pd.DataFrame(rows, columns=["date", "instrument", "score"])
    out = skeleton.merge(pred, on=["date", "instrument"], how="left")
    # 缺失分数填入当日截面中位数，避免固定值 0 偏离分布中心
    out["score"] = out["score"].replace([np.inf, -np.inf], np.nan)
    out["score"] = out.groupby("date")["score"].transform(
        lambda x: x.fillna(x.median()))
    # 极端情况：某天完全没有有效分（全部 NaN），退化为 0
    out["score"] = out["score"].fillna(0.0)
    return out[["date", "instrument", "score"]].reset_index(drop=True)

In [ ]:
import os


def export_model_json(ckpt, path):
    """把 PyTorch checkpoint 无损编码为官方要求的 JSON 训练产物。"""
    import base64
    import json

    encoded_state = {}
    for name, value in ckpt["state_dict"].items():
        array = value.detach().cpu().contiguous().numpy()
        encoded_state[name] = {
            "dtype": array.dtype.str,
            "shape": list(array.shape),
            "data_b64": base64.b64encode(array.tobytes()).decode("ascii"),
        }
    artifact = {
        "format_version": 1,
        "artifact_type": "bigalpha_pytorch_state_dict",
        "state_dict": encoded_state,
        "mean": np.asarray(ckpt["mean"], np.float32).tolist(),
        "std": np.asarray(ckpt["std"], np.float32).tolist(),
        "config": ckpt["config"],
        "best_epoch": ckpt.get("best_epoch"),
        "val_metrics": ckpt.get("val_metrics"),
        "metric_version": ckpt.get("metric_version", 1),
    }
    target = os.fspath(path)
    temporary = target + ".tmp"
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(artifact, handle, ensure_ascii=False, separators=(",", ":"))
    os.replace(temporary, target)
    print(f"[artifact] saved {target}", flush=True)
    return target


def load_model_json(path):
    """读取 JSON 训练产物并还原为 run_inference 使用的 checkpoint。"""
    import base64
    import json

    with open(path, "r", encoding="utf-8") as handle:
        artifact = json.load(handle)
    if artifact.get("format_version") != 1:
        raise ValueError(
            f"不支持的模型产物版本: {artifact.get('format_version')}"
        )
    state_dict = {}
    for name, item in artifact["state_dict"].items():
        raw = base64.b64decode(item["data_b64"])
        array = np.frombuffer(raw, dtype=np.dtype(item["dtype"])).copy()
        array = array.reshape(item["shape"])
        state_dict[name] = torch.from_numpy(array)
    return {
        "state_dict": state_dict,
        "mean": np.asarray(artifact["mean"], np.float32),
        "std": np.asarray(artifact["std"], np.float32),
        "config": artifact["config"],
        "best_epoch": artifact.get("best_epoch"),
        "val_metrics": artifact.get("val_metrics"),
        "metric_version": artifact.get("metric_version", 1),
    }

In [ ]:
def main(datasources, start_date, end_date):
    """平台评判入口：加载 JSON 训练产物，对注入的测试表推理。"""
    import dai
    import structlog

    logger = structlog.get_logger()
    infer_table = datasources["bar15m"]
    # 低频表优先从平台注入的 datasources 取，硬编码名仅作兜底
    lowfreq_table = datasources.get(
        "bar30m", CONFIG.get("lowfreq_table"))

    def query_fn(sql, filters):
        return dai.query(sql, filters=filters, compression=True).df()

    def instruments_query_fn(sd, ed):
        return dai.query(
            f"SELECT date, instrument FROM {CONFIG['instruments_table']}",
            filters={"date": [sd, ed]},
            compression=True,
        ).df()

    ckpt = load_model_json(CONFIG["artifact_path"])
    required = {"state_dict", "mean", "std", "config"}
    missing = required.difference(ckpt)
    if missing:
        raise KeyError(f"checkpoint 缺少字段: {sorted(missing)}")
    logger.info(
        "JSON 模型加载完成",
        best_epoch=ckpt.get("best_epoch"),
        val_metrics=ckpt.get("val_metrics"),
    )
    result = run_inference(
        ckpt,
        query_fn,
        infer_table,
        instruments_query_fn,
        start_date,
        end_date,
        lowfreq_table=lowfreq_table,
    )
    logger.info(
        "分数构建完成",
        rows=len(result),
        days=result["date"].nunique(),
        instruments=result["instrument"].nunique(),
    )
    return result


def local_evaluate(
    start_date="2023-10-01 00:00:00",
    end_date="2023-12-31 23:59:59",
):
    """仅供开发环境手动调用；平台评判不会调用此函数。"""
    from bigmodule import M

    datasources = {"bar15m": "bigalpha_2026_stock_bar15m"}
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    return M.bigalpha_eval._latest(factor_data=score_data, show=True)

In [ ]:
# 本地调试时手动取消下一行注释；正式提交保持注释。
# local_result = local_evaluate()

In [ ]:
"""平台端诊断：验证推理链路各环节是否正常。

在平台上运行此 cell 可提前发现表查询、模型加载、shape 对齐等问题，
避免提交后无日志失败。
"""
import os
import sys
import traceback

# ── 0. 环境 ──
print("=" * 60)
print("0. 环境检查")
print("=" * 60)
print(f"  Python: {sys.version}")
try:
    import torch
    print(f"  PyTorch: {torch.__version__}")
    print(f"  CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  CUDA device: {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"  PyTorch: ERROR ({e})")

try:
    import dai
    print(f"  dai: available")
except Exception as e:
    print(f"  dai: NOT AVAILABLE ({e})")

print(f"  artifact exists: {os.path.exists(CONFIG['artifact_path'])}")

# ── 1. 模型 JSON 加载 ──
print()
print("=" * 60)
print("1. 模型 JSON 加载")
print("=" * 60)
try:
    ckpt = load_model_json(CONFIG["artifact_path"])
    cfg = ckpt["config"]  # 训练时的配置，推理必须以此为准
    print(f"  format_version: {ckpt.get('format_version')}")
    print(f"  state_dict keys: {len(ckpt['state_dict'])}")
    print(f"  mean/std shape: {ckpt['mean'].shape}, {ckpt['std'].shape}")
    print(f"  lowfreq_table: {cfg.get('lowfreq_table')}")
    print(f"  lowfreq_mode: {cfg.get('lowfreq_mode')}")
    print(f"  bars_per_day: {cfg['bars_per_day']}")
    print(f"  lowfreq_bars_per_day: {cfg.get('lowfreq_bars_per_day')}")
    print(f"  model d_intra: {cfg['model']['d_intra']}")
    print(f"  ✓ JSON 加载成功")
except Exception as e:
    print(f"  ✗ 加载失败: {e}")
    traceback.print_exc()
    ckpt = None

# ── 2. 模型构建 + 权重加载 ──
if ckpt is not None:
    print()
    print("=" * 60)
    print("2. 模型构建 + 权重加载")
    print("=" * 60)
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = StockScorer(cfg).to(device)
        n_params = count_params(model)
        model.load_state_dict(ckpt["state_dict"])
        model.eval()
        print(f"  参数量: {n_params:,}")
        print(f"  has_lowfreq: {model.has_lowfreq}")
        print(f"  lowfreq_mode: {model.lowfreq_mode}")
        if model.lowfreq_mode == "concat":
            expected = cfg["bars_per_day"] + cfg["lowfreq_bars_per_day"]
            actual = model.intra.pos.shape[1]
            print(f"  IntradayEncoder pos bars: {actual} (expected {expected})")
            assert actual == expected, f"pos bar 数不匹配!"
        print(f"  ✓ 模型构建 + 权重加载成功")
    except Exception as e:
        print(f"  ✗ 失败: {e}")
        traceback.print_exc()
        model = None
else:
    model = None

# ── 3. Dummy 前向传播 ──
if model is not None:
    print()
    print("=" * 60)
    print("3. Dummy 前向传播")
    print("=" * 60)
    try:
        with torch.no_grad():
            N, L, B, F = 2, 60, cfg["bars_per_day"], len(cfg["feature_cols"])
            x = torch.randn(N, L, B, F, device=device)

            if model.has_lowfreq and model.lowfreq_mode == "concat":
                BL = cfg["lowfreq_bars_per_day"]
                x_low = torch.randn(N, L, BL, F, device=device)
                out = model(x, x_low)
                print(f"  双频 forward: {out.shape} (expected ({N},))  ✓")

                # 测回退路径（低频数据缺失时零补齐）
                out_fb = model(x)
                print(f"  回退 forward: {out_fb.shape} (expected ({N},))  ✓")
                H = model.encode_days(x)
                print(f"  encode_days(x) fallback: (N={H.shape[0]}, L={H.shape[1]}, d={H.shape[2]})  ✓")
            else:
                out = model(x)
                print(f"  单频 forward: {out.shape} (expected ({N},))  ✓")

    except Exception as e:
        print(f"  ✗ 失败: {e}")
        traceback.print_exc()

# ── 4. 真实数据查询测试 ──
print()
print("=" * 60)
print("4. 真实数据查询测试")
print("=" * 60)
try:
    import dai
    
    def test_query_fn(sql, filters):
        return dai.query(sql, filters=filters, compression=True).df()
    
    test_start = "2023-12-01 00:00:00"
    test_end   = "2023-12-07 23:59:59"
    
    # 4a. 查股票池
    print(f"  查询 instruments ({test_start} ~ {test_end}) ...")
    inst_df = test_query_fn(
        f"SELECT date, instrument FROM {cfg['instruments_table']}",
        {"date": [test_start, test_end]})
    instruments = sorted(inst_df["instrument"].unique().tolist())
    print(f"  instruments: {len(instruments)} 只")
    
    # 4b. 查高频表
    print(f"  查询 15m 表 ...")
    arrs = load_stock_arrays(
        test_query_fn, "bigalpha_2026_stock_bar15m", cfg,
        test_start, test_end, instruments[:10],
        bars_per_day=cfg["bars_per_day"])
    n_stocks = len(arrs)
    sample_ins = next(iter(arrs))
    sample_dates, sample_X, _ = arrs[sample_ins]
    print(f"  15m 表: {n_stocks} stocks, sample shape={sample_X.shape} (D,B,F)  ✓")
    
    # 4c. 查低频表（关键！）
    lf_table = cfg.get("lowfreq_table")
    if lf_table:
        print(f"  查询 30m 表 ({lf_table}) ...")
        try:
            arrs_low = load_stock_arrays(
                test_query_fn, lf_table, cfg,
                test_start, test_end, instruments[:10],
                bars_per_day=cfg["lowfreq_bars_per_day"])
            if arrs_low:
                sample_ins = next(iter(arrs_low))
                _, sample_Xl, _ = arrs_low[sample_ins]
                print(f"  30m 表: {len(arrs_low)} stocks, sample shape={sample_Xl.shape} (D,B,F)  ✓")
                
                # 检查日期对齐
                common = set(arrs.keys()) & set(arrs_low.keys())
                if common:
                    ins = next(iter(common))
                    d_high = set(arrs[ins][0])
                    d_low  = set(arrs_low[ins][0])
                    overlap = d_high & d_low
                    print(f"  日期对齐: {ins} — high={len(d_high)}d low={len(d_low)}d overlap={len(overlap)}d")
                    if len(overlap) == 0:
                        print(f"  ⚠ 警告: 双频日期无交集！回退路径会走零补齐")
                    else:
                        print(f"  ✓ 日期对齐正常")
            else:
                print(f"  ⚠ 30m 表返回空！将触发回退零补齐路径")
        except Exception as e:
            print(f"  ⚠ 30m 表查询失败: {e}")
            print(f"     将触发回退零补齐路径（shape 保护已就位）")
    else:
        print(f"  无 lowfreq_table 配置，单频模式")

except Exception as e:
    print(f"  ✗ 数据查询环节失败: {e}")
    traceback.print_exc()

# ── 5. 端到端小样本推理 ──
print()
print("=" * 60)
print("5. 端到端推理（小样本）")
print("=" * 60)
try:
    datasources = {
        "bar15m": "bigalpha_2026_stock_bar15m",
        "bar30m": cfg.get("lowfreq_table", ""),
    }
    result = main(datasources, "2023-12-01 00:00:00", "2023-12-07 23:59:59")
    print(f"  rows: {len(result)}")
    print(f"  days: {result['date'].nunique()}")
    print(f"  instruments: {result['instrument'].nunique()}")
    print(f"  score range: [{result['score'].min():.4f}, {result['score'].max():.4f}]")
    print(f"  score mean/std: {result['score'].mean():.4f} / {result['score'].std():.4f}")
    nulls = result["score"].isna().sum()
    print(f"  null scores: {nulls} ({nulls/len(result)*100:.2f}%)")
    print(f"  columns: {list(result.columns)}")
    if nulls == 0 and len(result) > 0:
        print(f"  ✓ 端到端推理成功")
    else:
        print(f"  ⚠ 存在空值或空结果")
except Exception as e:
    print(f"  ✗ 推理失败: {e}")
    traceback.print_exc()

print()
print("=" * 60)
print("诊断完成 —— 若全部 ✓ 则可提交")
print("=" * 60)